# 04 — Visualization

Generate publication-ready charts from the master DUI-by-state dataset.

Charts produced:
1. **Choropleth maps** (twitter_landscape) — fatality rate per 100M VMT, felony status, IID status, max speed limit, prior DWI %
2. **Scatter** (twitter_landscape) — alcohol consumption vs fatality rate, colored by region
3. **Ranked bars** (instagram_portrait) — top/bottom 10 states by fatality rate per VMT
4. **IID comparison** (twitter_landscape) — IID vs non-IID mean fatality rates

All outputs saved to `outputs/` with @unwelcomedata watermark.

In [ ]:
import sys
import os
from pathlib import Path

%matplotlib inline
import pandas as pd
import yaml

# Find project root by walking up from cwd until we find config.yaml
PROJECT = Path.cwd()
while not (PROJECT / "config.yaml").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.viz import (
    choropleth_map,
    scatter_chart,
    ranked_bar_chart,
    comparison_chart,
    line_chart,
    save_chart,
)

# Load config
with open(PROJECT / "config.yaml") as f:
    cfg = yaml.safe_load(f)

# Load master table
df = pd.read_parquet(PROJECT / "export" / "dui_by_state_v2.parquet")
print(f"Project root: {PROJECT}")
print(f"Master table: {df.shape[0]} states x {df.shape[1]} columns")
df.head(3)

## 1. Choropleth Maps

Flexible map function — swap `column` and `mode` to explore any measure.

In [ ]:
# --- Map 1: Alcohol fatality rate per 100M VMT (heat) ---

fig = choropleth_map(
    df,
    column="alcohol_fatality_rate_per_100m_vmt",
    title="Alcohol-Impaired Fatality Rate by State",
    subtitle="Deaths per 100 million vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    mode="heat",
    legend_title="Deaths per 100M VMT",
    preset="twitter_landscape",
    annotate=True,
)
save_chart(fig, cfg, "map_alcohol_fatality_rate_vmt", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 2: Felony vs misdemeanor (category) ---

# Create a clean felony label column
df["felony_label"] = df["first_offense_felony"].map({1.0: "Can be felony", 0.0: "Always misdemeanor"})
df.loc[df["felony_label"].isna(), "felony_label"] = "Unknown"

fig = choropleth_map(
    df,
    column="felony_label",
    title="First-Offense DUI: Felony Possible?",
    subtitle="States where first DUI can be charged as felony (e.g. with child in car or injury)",
    source="NCSL DUI/DWI criminal status laws",
    mode="category",
    category_colors={"Can be felony": "#DC2626", "Always misdemeanor": "#2563EB", "Unknown": "#E5E7EB"},
    legend_title="First-offense status",
    preset="twitter_landscape",
)
save_chart(fig, cfg, "map_felony_status", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 3: IID requirement (category) ---

df["iid_label"] = df["iid_all_offender"].map({1: "All offenders", 0: "Repeat/high-BAC only"})

fig = choropleth_map(
    df,
    column="iid_label",
    title="Ignition Interlock: All First Offenders?",
    subtitle="States requiring IID for all DUI offenders vs repeat/high-BAC only",
    source="IIHS, GHSA, NHTSA enforcement compilations",
    mode="category",
    category_colors={"All offenders": "#16A34A", "Repeat/high-BAC only": "#D97706"},
    legend_title="IID requirement",
    preset="twitter_landscape",
)
save_chart(fig, cfg, "map_iid_status", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 4: Max speed limit (heat) ---

fig = choropleth_map(
    df,
    column="max_speed_limit_mph",
    title="Maximum Posted Speed Limit by State",
    subtitle="Rural interstate max speed (mph)",
    source="IIHS, August 2026",
    mode="heat",
    cmap=["#DBEAFE", "#60A5FA", "#2563EB", "#7C3AED", "#4C1D95"],
    legend_title="Max speed (mph)",
    preset="twitter_landscape",
    annotate=True,
)
save_chart(fig, cfg, "map_max_speed_limit", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 5: Prior DWI % (heat) ---

fig = choropleth_map(
    df,
    column="pct_impaired_with_prior_dwi",
    title="Repeat Offenders in Fatal Crashes",
    subtitle="% of impaired drivers in fatal crashes with a prior DWI conviction",
    source="NHTSA FARS 2024",
    mode="heat",
    cmap=["#FEF3C7", "#FBBF24", "#D97706", "#DC2626", "#7F1D1D"],
    legend_title="% with prior DWI",
    preset="twitter_landscape",
    annotate=True,
)
save_chart(fig, cfg, "map_prior_dwi_pct", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

## 2. Scatter — Consumption vs Fatality Rate

In [ ]:
fig = scatter_chart(
    df,
    x="ethanol_per_capita_gallons_2022",
    y="alcohol_fatality_rate_per_100m_vmt",
    color_by="region",
    title="Alcohol Consumption vs Impaired-Driving Deaths",
    subtitle="Each dot is a state. Per-VMT fatality rate controls for driving exposure.",
    source="NIAAA consumption 2022, NHTSA FARS 2024, FHWA VMT 2022",
    xlabel="Per capita ethanol (gallons, 2022)",
    ylabel="Alcohol fatalities per 100M VMT",
    annotate=True,
    preset="twitter_landscape",
)
save_chart(fig, cfg, "scatter_consumption_vs_fatality", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

## 3. Ranked Bars — Top/Bottom 10 by Fatality Rate (per VMT)

In [ ]:
fig = ranked_bar_chart(
    df,
    x="state_name",
    y="alcohol_fatality_rate_per_100m_vmt",
    title="Worst & Best States for Impaired-Driving Deaths",
    subtitle="Alcohol fatalities per 100M vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    top_n=10,
    bottom_n=10,
    value_fmt="{:.2f}",
    preset="instagram_portrait",
)
save_chart(fig, cfg, "ranked_top_bottom_10_vmt", preset="instagram_portrait", add_watermark="@unwelcomedata", close=False)
fig

## 4. IID vs Non-IID Comparison

In [ ]:
# Create IID group label
df["iid_group"] = df["iid_all_offender"].map({1: "IID for all offenders", 0: "No universal IID"})

fig = comparison_chart(
    df,
    group_col="iid_group",
    value_col="alcohol_fatality_rate_per_100m_vmt",
    title="Does Mandatory IID Reduce Impaired-Driving Deaths?",
    subtitle="Mean alcohol fatality rate per 100M VMT by IID policy (error bars = standard error)",
    source="NHTSA FARS 2024, FHWA VMT 2022, IIHS/GHSA enforcement data",
    colors={"IID for all offenders": "#16A34A", "No universal IID": "#DC2626"},
    value_fmt="{:.2f}",
    preset="twitter_landscape",
)
save_chart(fig, cfg, "comparison_iid_vs_no_iid", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

## Quick exploration

One-liner functions to explore any columns. Just call and view inline.

In [ ]:
def quick_map(column, title=None, mode="heat", **kwargs):
    """Choropleth any column. mode='heat' for numeric, 'category' for discrete."""
    title = title or column.replace('_', ' ').title()
    return choropleth_map(df, column=column, title=title, mode=mode, preset='twitter_landscape', annotate=True, **kwargs)


def quick_scatter(x, y, title=None, color_by='region', **kwargs):
    """Scatter any two numeric columns. Colored by region by default."""
    title = title or f"{x.replace('_',' ').title()} vs {y.replace('_',' ').title()}"
    return scatter_chart(df, x=x, y=y, color_by=color_by, title=title, preset='twitter_landscape', annotate=True, **kwargs)


def quick_bars(y, x='state_name', title=None, top_n=10, bottom_n=0, **kwargs):
    """Ranked bar chart. Shows top_n (and optionally bottom_n) states."""
    title = title or f"Top {top_n} States by {y.replace('_',' ').title()}"
    return ranked_bar_chart(df, x=x, y=y, title=title, top_n=top_n, bottom_n=bottom_n, preset='instagram_portrait', **kwargs)


def quick_compare(group_col, value_col, title=None, **kwargs):
    """Compare group means for any grouping column vs any numeric column."""
    title = title or f"{value_col.replace('_',' ').title()} by {group_col.replace('_',' ').title()}"
    return comparison_chart(df, group_col=group_col, value_col=value_col, title=title, preset='twitter_landscape', **kwargs)


def quick_trend(x='year', y='impaired_fatalities', data=None, title=None, **kwargs):
    """Line trend chart. Pass a different DataFrame via data= if needed."""
    title = title or f"{y.replace('_',' ').title()} over {x.replace('_',' ').title()}"
    src = data if data is not None else df
    return line_chart(src, x=x, y=y, title=title, preset='twitter_landscape', **kwargs)


def quick_bubble_map(color_col, size_col, title=None, **kwargs):
    """Map with state fill = color_col, bubble overlay = size_col."""
    title = title or f"{color_col.replace('_',' ').title()} (fill) + {size_col.replace('_',' ').title()} (bubble)"
    fig = choropleth_map(df, column=color_col, title=title, mode='heat', preset='twitter_landscape', annotate=False, **kwargs)
    # Overlay bubbles using lat/lng from the dataframe
    ax = fig.axes[0]
    import numpy as np
    from src.viz import PALETTE
    plot_df = df[['state_abbr', 'lat', 'lng', size_col]].dropna()
    sizes = (plot_df[size_col] - plot_df[size_col].min()) / (plot_df[size_col].max() - plot_df[size_col].min())
    # We need projected coords — use the geo data instead
    import geopandas as gpd
    from pathlib import Path
    shp = Path('data/raw/geo/cb_2022_us_state_20m.shp')
    geo = gpd.read_file(shp)
    geo = geo[~geo['STATEFP'].isin({'60','66','69','72','78'})].to_crs('EPSG:5070')
    geo['cx'] = geo.geometry.centroid.x
    geo['cy'] = geo.geometry.centroid.y
    merged = geo[['STUSPS','cx','cy']].merge(df[['state_abbr', size_col]], left_on='STUSPS', right_on='state_abbr')
    vals = merged[size_col].fillna(0)
    norm_sizes = 20 + 300 * (vals - vals.min()) / (vals.max() - vals.min())
    ax.scatter(merged['cx'], merged['cy'], s=norm_sizes, alpha=0.6, color='#E76F51', edgecolors='white', linewidths=0.5, zorder=5)
    return fig


def quick_bivariate_map(x_col, y_col, title=None, n_bins=3):
    """Bivariate choropleth — 3x3 color grid showing two variables."""
    import geopandas as gpd
    import numpy as np
    from pathlib import Path
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from src.viz import PRESETS, STYLE, _fig_for_preset, _add_title_block, _shift_alaska_hawaii, _load_states_geo
    title = title or f"{x_col.replace('_',' ').title()} vs {y_col.replace('_',' ').title()}"
    # 3x3 bivariate color grid
    biv_colors = [
        ['#E8E8E8', '#B8D6BE', '#2A9D8F'],  # y_low: x_low -> x_high
        ['#DFC27D', '#B5A068', '#6D8A6D'],  # y_mid
        ['#E76F51', '#C45A3C', '#264653'],  # y_high: x_low -> x_high
    ]
    data = df[['state_fips', x_col, y_col]].dropna().copy()
    data['state_fips'] = data['state_fips'].astype(str).str.zfill(2)
    data['_xbin'] = pd.qcut(data[x_col], n_bins, labels=False, duplicates='drop')
    data['_ybin'] = pd.qcut(data[y_col], n_bins, labels=False, duplicates='drop')
    data['_color'] = data.apply(lambda r: biv_colors[int(min(r['_ybin'], n_bins-1))][int(min(r['_xbin'], n_bins-1))], axis=1)
    geo = _load_states_geo()
    geo = _shift_alaska_hawaii(geo)
    geo = geo.merge(data[['state_fips','_color']], left_on='STATEFP', right_on='state_fips', how='left')
    geo['_color'] = geo['_color'].fillna('#E5E7EB')
    fig, ax = _fig_for_preset('twitter_landscape')
    ax.set_axis_off()
    for color in geo['_color'].unique():
        geo[geo['_color'] == color].plot(ax=ax, color=color, edgecolor='white', linewidth=0.5)
    # Legend grid
    legend_ax = fig.add_axes([0.82, 0.15, 0.12, 0.12])
    for yi in range(n_bins):
        for xi in range(n_bins):
            legend_ax.add_patch(mpatches.Rectangle((xi, yi), 1, 1, color=biv_colors[yi][xi]))
    legend_ax.set_xlim(0, n_bins)
    legend_ax.set_ylim(0, n_bins)
    legend_ax.set_xlabel(x_col.replace('_',' ').title()[:15] + ' →', fontsize=7)
    legend_ax.set_ylabel(y_col.replace('_',' ').title()[:15] + ' →', fontsize=7)
    legend_ax.set_xticks([])
    legend_ax.set_yticks([])
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left')
    fig.subplots_adjust(left=0.02, right=0.95, top=0.90, bottom=0.05)
    return fig


print('Quick functions ready: quick_map, quick_scatter, quick_bars, quick_compare, quick_trend, quick_bubble_map, quick_bivariate_map')

In [ ]:
def cols():
    """Print all columns in the master table, grouped by type."""
    numeric = [c for c in df.columns if df[c].dtype in ('float64', 'int64')]
    categorical = [c for c in df.columns if df[c].dtype == 'object']
    print(f'=== NUMERIC ({len(numeric)}) ===')
    for c in numeric:
        print(f'  {c}')
    print(f'\n=== CATEGORICAL ({len(categorical)}) ===')
    for c in categorical:
        vals = df[c].nunique()
        print(f'  {c}  ({vals} unique)')

cols()

In [ ]:
# --- Try it: consumption vs arrest rate ---
quick_scatter('ethanol_per_capita_gallons_2022', 'dui_arrest_rate_per_100k_reporting',
              title='Alcohol Consumption vs DUI Arrest Rate')

In [ ]:
# More examples (uncomment any):
# quick_map('dui_arrest_rate_per_100k', title='DUI Arrest Rate per 100k')
# quick_map('ethanol_per_capita_gallons_2022', title='Per Capita Alcohol Consumption')
# quick_scatter('dui_arrest_rate_per_100k', 'alcohol_fatality_rate_per_100m_vmt')
# quick_bars('dui_arrest_rate_per_100k', top_n=10, bottom_n=10)
# quick_compare('iid_group', 'dui_arrest_rate_per_100k')
# quick_map('pct_suspended', title='% Drivers on Suspended License in Fatal Crashes')

In [ ]:
# Two-variable map: bubble overlay
quick_bubble_map('ethanol_per_capita_gallons_2022', 'pct_alcohol_nhtsa_imputed',
                 title='Consumption (fill) + % Traffic Deaths from Alcohol (bubble)')

In [ ]:
# Two-variable map: bivariate 3x3 grid
quick_bivariate_map('ethanol_per_capita_gallons_2022', 'pct_alcohol_nhtsa_imputed',
                    title='Bivariate: Consumption vs % Traffic Deaths from Alcohol')